# RAG with Feast

#### Install required dependencies

In [ ]:
%pip install --quiet pymilvus feast transformers sentence-transformers faiss-cpu datasets

#### Retrieve test dataset and chunk it

In [ ]:
from datasets import load_dataset

dataset = load_dataset("facebook/wiki_dpr", "psgs_w100.nq.exact", with_index=False)

In [ ]:
def chunk_dataset(examples, chunk_size=100, overlap=20):
    all_chunks = []
    all_ids = []
    all_titles = []

    for i, text in enumerate(examples['text']):  # Iterate over texts in the batch
        words = text.split()
        chunks = []
        for j in range(0, len(words), chunk_size - overlap):
            chunk_words = words[j:j + chunk_size]
            if len(chunk_words) < 20:
                continue
            chunk_text_value = ' '.join(chunk_words)  # Store the chunk text
            chunks.append(chunk_text_value)
            all_ids.append(f"{examples['id'][i]}_{j}")  # Unique ID for the chunk
            all_titles.append(examples['title'][i])

        all_chunks.extend(chunks)

    return {'id': all_ids, 'title': all_titles, 'text': all_chunks}


chunked_dataset = dataset['train'].map(
    chunk_dataset,
    batched=True,
    remove_columns=dataset['train'].column_names,
    num_proc=1
)

#### Define model and generate embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

sentences = chunked_dataset["text"]

# load pretrained sentence transformer model and create embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(sentences, show_progress_bar=True, batch_size=64, device="cuda")

print(f"Generated embeddings of shape: {embeddings.shape}")


#### Connect to Milvus vector database

In [ ]:
from pymilvus import connections

# Connect to milvus 
connections.connect(alias="default", host="vectordb-milvus.milvus.svc.cluster.local", port="19530", timeout=10, user="root", password="Milvus")

# Check connection status 
print("Connected:", connections.has_connection(alias="default"))

#### Create collection in Milvus

In [ ]:
from pymilvus import FieldSchema, CollectionSchema, DataType, Collection, utility

# Drop the old collection if it exists
if utility.has_collection("wiki_sentences"):
    Collection("wiki_sentences", using="default").drop()

# Define schema
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384)
]
schema = CollectionSchema(fields)
collection = Collection(name="wiki_sentences", schema=schema, using="default")

# Create index
collection.create_index(
    field_name="embedding",
    index_params={"index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 64}}
)

collection.load()

#### Insert embeddings into Milvus

In [ ]:
ids = list(range(len(sentences)))
vectors = embeddings.tolist()

collection.insert([ids, vectors])
collection.flush()

print(f"Inserted {len(ids)} wiki_dpr embeddings into Milvus.")

#### Using test dataset to create parquet file as historical data source

In [ ]:
import pandas as pd
from datetime import datetime

# Create DataFrame
df = pd.DataFrame({
    "passage_id": list(range(len(sentences))),
    "passage_text": sentences,
    "embedding": pd.Series(list(embeddings), dtype=object),
    "event_timestamp": [datetime.utcnow() for _ in sentences],
})

# Save to Parquet
df.to_parquet("feast_setup/data/wiki_dpr.parquet", index=False)
print("Saved to wiki_dpr.parquet")

#### Move into feast_setup directory and create feast objects

In [ ]:
%cd feast_setup
!feast apply
%cd kfto_feast_rag

#### Initialize Milvus Vector Store and FeastRAGRetriever

In [ ]:
from feast import FeatureStore
from feast_rag_retriever import FeastVectorStore, FeastRAGRetriever, RAGPipeline
# Setup
store = FeatureStore(repo_path="")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

vector_store = FeastVectorStore(
    store=store,
    rag_view=wiki_passage_content,
)

retriever = FeastRAGRetriever(embedding_model, vector_store)
rag_pipeline = RAGPipeline(retriever)

#### Generate response for provided query using top_k retrieved context

In [ ]:
# Ask a question
query = "What did Socrates say in his trial?"
answer = rag_pipeline.generate(query, top_k=5)
print("Generated Answer:", answer)